# 04 — 기본값(default) 모델 성능 비교: 회귀 / 투스테이지 / ZIT

5개 트리 모델(lgbm·xgb·catboost·et·rf) + **ZIT 16종**(4 base + 백엔드 믹스 12)을 **순수 라이브러리 기본값**으로 돌려 성능만 비교한다. HPO·후처리 없음.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_train_data.csv`
- **출력**: `4_output/0_baseline/default_compare/results.csv` (**46행** = reg 5 + two_stage 25 + zit 16) — §5에서 한 번에 저장
- **전처리**: 트리 공통 `PP_FIXED` + `CLIP_Y_EXTREME` + meta features — §2에서 1회, 세 모드가 같은 결과 공유 (ZIT도 die-level 행렬로 재사용)
- **모드 3종** (§4 학습 → §5 RMSE 집계):
  - **기본 회귀** (5건): die-level full-y 학습 → die→unit mean → RMSE
  - **투스테이지** (clf 5 × reg 5 = 25건): `P(Y>0) × E[Y|Y>0]` die 곱셈 → unit mean → RMSE
  - **ZIT 4 base**: zero-inflated Tweedie 단일 모델 2×2 — φ(우리 Pearson / 논문충실 EQL) × bag 제약. ζ는 profile likelihood 추정. 집계는 일반 ZIT=mean·BagZIT=sum
  - **ZIT 백엔드 믹스 12종** (§4c): 위 4 base × {π=CatBoost·μ=LGBM / π=LGBM·μ=CatBoost / 둘 다 CatBoost} — φ는 모두 LGBM. ζ*는 §4b baseline 차용. CatBoost μ=Tweedie·π=CrossEntropy(나머지 HP는 CatBoost 기본값)
- **파라미터**: 전 모델 라이브러리 기본값. HP 안 박음. 분류기 imbalance 보정도 OFF(=라이브러리 기본). ZIT 내부 μ/π/φ LightGBM도 라이브러리 기본값(ζ만 추정, n_em_iters=10).
- **후처리 없음**: die→unit `mean`/`sum` 집계만 (τ/position/zero_clip/집계선택 전부 미적용)
- **구성**: §1 설정 → §2 데이터·전처리 → §3 공통 헬퍼·fold → §4 학습(트리 15회 refit → ZIT 4종 → §4c ZIT 믹스 12종) → §5 결과·저장

In [1]:
import os, sys

# Google Drive 파일 ID (Colab 자동 다운로드용, 로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'   # preprocessing.zip
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (zit.py φ 가드 반영본 재업로드 필요)

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/models.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리(2_preprocessing) + 모델링(3_modeling)을 패키지 접두사로 import 하도록 경로 추가
for _d in (os.path.join(PROJECT_ROOT, '2_preprocessing'), os.path.join(PROJECT_ROOT, '3_modeling')):
    if _d not in sys.path:
        sys.path.insert(0, _d)

from modules import preprocess, hpo          # 전처리 래퍼 + refit (HPO 안 씀, refit_best/refit_clf_best만)
from meta_features import add_meta_features   # die_xy / position 메타피처

# ZIT 4종 (zero-inflated Tweedie + LightGBM EM) — §4b ZIT 학습용
from modules.zit import (
    ZITboostRegressor, ZITboostEQLRegressor,
    BagZITboostRegressor, BagZITEQLRegressor,
)
from sklearn.model_selection import KFold     # unit 단위 fold (트리 refit_best와 동일 방식, ZIT refit 공용)

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'reg 모델: {hpo._models.AVAILABLE_MODELS}')
print(f'clf 모델: {hpo._models.CLF_AVAILABLE_MODELS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
reg 모델: ['lgbm', 'xgb', 'catboost', 'et', 'rf', 'enet', 'zitboost']
clf 모델: ['lgbm', 'xgb', 'catboost', 'et', 'rf']


## 1. 설정 — 모델 / 전처리 고정값 / 기본 파라미터 (트리 + ZIT)

트리 5종 + ZIT 4종 설정을 한곳에 모은다. `PP_FIXED`·`CLIP_Y_EXTREME`은 현재 트리 노트북에 고정된 값 그대로.
기본 파라미터는 **라이브러리 기본값 + 재현(random_state)·병렬(n_jobs)·로그억제만**. HP는 일절 안 박는다.
분류기 imbalance 보정은 각 라이브러리의 '가중치 없음' 기본값을 명시로 박아 `refit_clf_best`의 자동 보정(setdefault)을 무력화한다.
ZIT는 내부 μ/π/φ LightGBM 3개를 라이브러리 기본값(`LGBM_DEFAULTS`)으로 두고, ζ(Tweedie power)만 profile likelihood로 추정(`ZETA_GRID`).

In [2]:
N_FOLDS = 5
N_JOBS  = -1   # 모델 학습 병렬도 (단독 실행이면 14로). strategy_common §8

MODELS = ['lgbm', 'xgb', 'catboost', 'et', 'rf']

CLIP_Y_EXTREME = True   # train y의 max(=1.0, 1건)를 두 번째로 큰 값으로 clip (학습 입력 안정화)

# 트리 공통 고정 전처리 (strategy_common §1)
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

OUT_DIR = os.path.join(OUTPUT_DIR, '0_baseline', 'default_compare')
os.makedirs(OUT_DIR, exist_ok=True)


def reg_default_params(name):
    # 회귀: 라이브러리 기본값 + 재현/병렬/로그억제만 (HP 미설정)
    if name == 'lgbm':     return dict(random_state=SEED, n_jobs=N_JOBS, verbose=-1)
    if name == 'xgb':      return dict(random_state=SEED, n_jobs=N_JOBS, verbosity=0)
    if name == 'catboost': return dict(random_seed=SEED, thread_count=N_JOBS, verbose=False, allow_writing_files=False)
    if name == 'et':       return dict(random_state=SEED, n_jobs=N_JOBS)
    if name == 'rf':       return dict(random_state=SEED, n_jobs=N_JOBS)
    raise KeyError(name)


def clf_default_params(name):
    # 분류: 위와 동일 + imbalance 보정 OFF (라이브러리 '가중치 없음' 기본값을 명시 → 자동 보정 무력화)
    if name == 'lgbm':     return dict(random_state=SEED, n_jobs=N_JOBS, verbose=-1, scale_pos_weight=1.0)
    if name == 'xgb':      return dict(random_state=SEED, n_jobs=N_JOBS, verbosity=0, scale_pos_weight=1.0)
    if name == 'catboost': return dict(random_seed=SEED, thread_count=N_JOBS, verbose=False, allow_writing_files=False, auto_class_weights=None)
    if name == 'et':       return dict(random_state=SEED, n_jobs=N_JOBS, class_weight=None)
    if name == 'rf':       return dict(random_state=SEED, n_jobs=N_JOBS, class_weight=None)
    raise KeyError(name)


# --- ZIT 4종 설정 (05_zit_default와 동일 깡통 철학) ---
N_EM_ITERS = 10   # Generalized EM 반복 수 (Algorithm 1)
# ζ는 논문처럼 추정: 각 후보로 EM→train 로그우도 비교→최대 ζ* 선택 (Algorithm 2)
ZETA_GRID  = [round(z, 2) for z in np.arange(1.1, 1.85, 0.1)]   # [1.1, ..., 1.8] (8개)

# ZIT 내부 μ/π/φ LightGBM 3개 — 전부 LightGBM '라이브러리 기본값'으로 명시 (트리 깡통과 동일 철학)
LGBM_DEFAULTS = dict(
    mu_n_estimators=100, mu_learning_rate=0.1, mu_num_leaves=31, mu_max_depth=-1,
    mu_min_child_samples=20, mu_subsample=1.0, mu_colsample_bytree=1.0,
    mu_reg_alpha=0.0, mu_reg_lambda=0.0,
    pi_n_estimators=100, pi_learning_rate=0.1, pi_num_leaves=31, pi_max_depth=-1,
    pi_min_child_samples=20,
    phi_n_estimators=100, phi_learning_rate=0.1, phi_num_leaves=31, phi_max_depth=-1,
    phi_min_child_samples=20,
)

# 4종 스펙 (라벨, 클래스, bag 여부, die→unit 집계, φ 방식)
#   일반 ZIT: die가 unit health를 broadcast 학습 → mean / BagZIT: die가 unit health 몫을 분배 학습 → sum
#   phi: ζ profile은 일반 ZIT(broadcast y, 스케일 일치)에서만 돌리고 BagZIT는 같은 φ의 ζ*를 차용
ZIT_SPECS = [
    {'label': 'zit_pearson',    'cls': ZITboostRegressor,    'bag': False, 'agg': 'mean', 'phi': 'pearson'},  # zit 우리버전
    {'label': 'zit_eql',        'cls': ZITboostEQLRegressor, 'bag': False, 'agg': 'mean', 'phi': 'eql'},      # zit 논문충실
    {'label': 'bagzit_pearson', 'cls': BagZITboostRegressor, 'bag': True,  'agg': 'sum',  'phi': 'pearson'},  # bag 우리버전
    {'label': 'bagzit_eql',     'cls': BagZITEQLRegressor,   'bag': True,  'agg': 'sum',  'phi': 'eql'},      # bag 논문충실
]

print('MODELS:', MODELS)
print('N_FOLDS:', N_FOLDS, '| N_JOBS:', N_JOBS)
print('ZIT_SPECS:', [s['label'] for s in ZIT_SPECS], '| ZETA_GRID:', ZETA_GRID, '| N_EM_ITERS:', N_EM_ITERS)
print('OUT_DIR:', OUT_DIR)

MODELS: ['lgbm', 'xgb', 'catboost', 'et', 'rf']
N_FOLDS: 5 | N_JOBS: -1
ZIT_SPECS: ['zit_pearson', 'zit_eql', 'bagzit_pearson', 'bagzit_eql'] | ZETA_GRID: [np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(1.7), np.float64(1.8)] | N_EM_ITERS: 10
OUT_DIR: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\0_baseline\default_compare


## 2. 데이터 로드 + 고정 전처리 (1회) — 트리·ZIT 공유

`preprocess.run`(cleaning + spatial imputation + outlier) → meta features. 전 모델·전 모드가 같은 전처리 결과를 공유한다.
ZIT는 die-level로 학습하므로, 같은 전처리 결과를 `X_train/val/test`(np.float64 행렬) + die→unit 매핑(`uid_*_die`) + unit health를 4 die에 broadcast한 `y_train_die`로도 만들어 둔다.

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y 극단값(1.0 1건)만 두 번째로 큰 값으로 clip
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f} clip, {n_clipped}개 샘플')

# 고정 전처리 1회
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# 메타피처 (트리: position raw 정수 + die_x/die_y 연속형)
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

# unit-level 정답 (index=ufs_serial)
y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# --- ZIT용 die-level 행렬/매핑/타깃 (같은 전처리 결과 재사용 — 결측 채워져 NaN 없음) ---
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

# die-level 정답: 각 die에 자기 unit health를 broadcast (full-y)
y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)
assert not np.isnan(y_train_die).any(), 'y_train_die NaN — train die의 unit이 y에 없음'

print(f'[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  xs_train: {xs_train.shape}, xs_val: {xs_val.shape}, xs_test: {xs_test.shape}')
print(f'  X_train: {X_train.shape} (NaN={int(np.isnan(X_train).sum())}), X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 -> 0.097417 clip, 1개 샘플
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 894개
    컬럼: 921 → 894 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 330개, 잔여: 564개
    컬럼: 894 → 564 (330개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, 

## 3. 공통 헬퍼 · fold (세 모드 공유)

die→unit 집계·unit RMSE·split별 매핑(UID)·정답(YK)·unit 단위 fold(FOLDS)·ZIT 모델 빌더를 한곳에 정의해 §4·§5가 공유한다.
- 집계: 기본 회귀/투스테이지/일반 ZIT = **mean**, BagZIT = **sum** (die가 unit health 몫을 분배 학습하므로)
- fold: `refit_best`(트리 내부)와 동일하게 **unit ID 단위** KFold(shuffle + SEED) → 트리·ZIT OOF가 같은 분할 위에서 나옴 (같은 unit의 4 die가 train/val에 섞이면 leakage)

In [4]:
# --- die→unit 집계 (how='mean': 회귀/투스테이지/일반 ZIT, how='sum': BagZIT) ---
def unit_agg(uid_die, die_pred, how='mean'):
    g = pd.DataFrame({KEY_COL: uid_die, 'v': np.asarray(die_pred)}).groupby(KEY_COL, sort=False)['v']
    return g.mean() if how == 'mean' else g.sum()

# 예측 Series를 정답 index 순서에 맞춰 unit RMSE
def rmse_unit(pred_s, y_s):
    p = pred_s.loc[y_s.index]
    return float(np.sqrt(np.mean((p.values - y_s.values) ** 2)))

# split별 die→unit 매핑 key + 정답 (reg/two_stage die 예측 집계용)
UID = {'oof': xs_train[KEY_COL].values, 'val': xs_val[KEY_COL].values, 'test': xs_test[KEY_COL].values}
YK  = {'oof': y_train_unit_s, 'val': y_val_unit_s, 'test': y_test_unit_s}

# unit ID 단위 KFold — refit_best(_make_unit_folds)와 동일 방식(ufs_serial 등장순서 + SEED) → 트리·ZIT OOF가 같은 분할
unique_units = y_train_unit_s.index.values
FOLDS = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(unique_units))

# --- ZIT 모델 빌더/적합 헬퍼 ---
def make_zit(spec, zeta):
    return spec['cls'](
        zeta=zeta, n_em_iters=N_EM_ITERS,
        random_state=SEED, n_jobs=N_JOBS, verbose=-1, device='cpu', **LGBM_DEFAULTS,
    )

def fit_zit(model, Xtr, ytr, uid_tr, is_bag):
    if is_bag:
        model.fit(Xtr, ytr, unit_id=uid_tr)   # BagZIT: unit_id 필수
    else:
        model.fit(Xtr, ytr)
    return model

print(f'[helpers] FOLDS={len(FOLDS)} (unit 단위) · UID/YK splits={list(UID)} · unit_agg/rmse_unit/make_zit/fit_zit 준비')

[helpers] FOLDS=5 (unit 단위) · UID/YK splits=['oof', 'val', 'test'] · unit_agg/rmse_unit/make_zit/fit_zit 준비


## 4. 모델 학습 — (4a) 트리 15회 refit → (4b) ZIT 4종

(4a) 트리: 모델 5개 × {기본회귀(full y) · 분류 P(Y>0) · 투스테이지 회귀 E[Y|Y>0]} = **15회** 5-fold refit → die 예측 `cache`.
(4b) ZIT: 2×2 스펙을 ζ profile(일반 ZIT) → ζ* 차용(BagZIT) → 5-fold refit → unit 예측 `zit_cache`.
die 예측을 캐싱해 두면 투스테이지 25조합·ZIT 4종 RMSE를 §5에서 재학습 없이 집계만으로 만든다.

> RF/ExtraTrees 회귀는 라이브러리 기본값 `max_features=1.0`(전 피처)이라 다소 느릴 수 있음 — 의도된 기본값.
> 참고 시간: 트리 refit은 모델별 편차 큼(catboost/et/rf가 김), ZIT 4종은 일반 ZIT의 ζ profile(8 후보) 포함 대략 1~1.5h.

In [5]:
# (4a) 트리 15회 refit — 모델 5개 × {기본회귀(full y) · 분류 P(Y>0) · 투스테이지 회귀 E[Y|Y>0]=y>0 only}
#   die-level 예측을 cache에 저장 → 투스테이지 25조합은 §5에서 재학습 없이 곱셈만으로 만든다.
cache = {}   # cache[name] = {'basic':{split:die}, 'clf':{split:die}, 'tsreg':{split:die}}
t_all = time.time()

for name in MODELS:
    t0 = time.time()
    print(f'\n===== {name} =====')

    # (1) 기본 회귀 — full y, die broadcast
    basic = hpo.refit_best(
        xs_train, xs_val, xs_test, ys_input['train'], feat_cols_clean,
        model_name=name, best_params=reg_default_params(name),
        n_folds=N_FOLDS, already_resolved=True, n_jobs=None,
    )
    # (2) 투스테이지 분류 — P(Y>0)
    clf = hpo.refit_clf_best(
        xs_train, xs_val, xs_test, ys_input['train'], feat_cols_clean,
        model_name=name, best_params=clf_default_params(name),
        n_folds=N_FOLDS, already_resolved=True, n_jobs=None,
    )
    # (3) 투스테이지 회귀 — E[Y|Y>0], y>0 only
    tsreg = hpo.refit_best(
        xs_train, xs_val, xs_test, ys_input['train'], feat_cols_clean,
        model_name=name, best_params=reg_default_params(name),
        n_folds=N_FOLDS, y_positive_only=True, already_resolved=True, n_jobs=None,
    )

    cache[name] = {
        'basic': {'oof': basic['oof_pred_die'], 'val': basic['val_pred_die'], 'test': basic['test_pred_die']},
        'clf':   {'oof': clf['oof_proba_die'],  'val': clf['val_proba_die'],  'test': clf['test_proba_die']},
        'tsreg': {'oof': tsreg['oof_pred_die'], 'val': tsreg['val_pred_die'], 'test': tsreg['test_pred_die']},
    }
    print(f'[{name}] done ({time.time()-t0:.0f}s)')

print(f'\n[트리 refit 완료] {time.time()-t_all:.0f}s')


===== lgbm =====
[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237
[clf refit fold 1/5] tr_units=20949, vl_units=5238, pos_ratio=0.290
[clf refit fold 2/5] tr_units=20949, vl_units=5238, pos_ratio=0.294
[clf refit fold 3/5] tr_units=20950, vl_units=5237, pos_ratio=0.293
[clf refit fold 4/5] tr_units=20950, vl_units=5237, pos_ratio=0.292
[clf refit fold 5/5] tr_units=20950, vl_units=5237, pos_ratio=0.291
[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237
[lgbm] done (32s)

===== xgb =====
[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5

In [6]:
# (4b) ZIT 4종 fit — 일반 ZIT는 ζ profile likelihood로 ζ* 결정, BagZIT는 같은 φ의 ζ* 차용 → 5-fold refit
zit_cache = {}      # label -> {'zeta_star', 'oof', 'val', 'test'(unit-level Series)}
zeta_by_phi = {}    # phi -> ζ* : 일반 ZIT가 profile로 결정, 같은 φ의 BagZIT가 차용
t_all = time.time()

for spec in ZIT_SPECS:
    label, is_bag, agg, phi = spec['label'], spec['bag'], spec['agg'], spec['phi']
    print(f'\n===== {label} (bag={is_bag}, agg={agg}, phi={phi}) =====')

    # (1) ζ 결정
    #   - 일반 ZIT: profile likelihood — die가 broadcast y를 직접 학습하므로 score_loglik 스케일 일치
    #   - BagZIT  : 같은 φ의 일반 ZIT ζ* 차용 — BagZIT은 die가 unit의 ¼ 몫을 학습(μ≈unit/4)해
    #               broadcast-y profile이 스케일 불일치 → 부정확한 profile 대신 일치하는 일반 ZIT ζ* 사용
    if not is_bag:
        t0 = time.time()
        ll_by_zeta = {}
        for z in ZETA_GRID:
            m = fit_zit(make_zit(spec, z), X_train, y_train_die, uid_train_die, is_bag)
            ll_by_zeta[z] = m.score_loglik(X_train, y_train_die)
        zeta_star = max(ll_by_zeta, key=ll_by_zeta.get)
        zeta_by_phi[phi] = zeta_star            # 같은 φ의 BagZIT가 차용
        print(f'  [ζ*] {zeta_star:.2f} (profile, train loglik 최대) | {time.time()-t0:.0f}s')
    else:
        zeta_star = zeta_by_phi[phi]            # 대응 일반 ZIT(zit_{phi})의 ζ* 차용
        print(f'  [ζ*] {zeta_star:.2f} (zit_{phi} ζ* 차용)')

    # (2) ζ* 5-fold refit — die 예측 (1-π)μ → oof/val/test (val/test는 fold 평균)
    oof_die  = np.full(len(X_train), np.nan)
    val_die  = np.zeros(len(X_val))
    test_die = np.zeros(len(X_test))
    t1 = time.time()
    for tr_uidx, vl_uidx in FOLDS:
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask  = np.isin(uid_train_die, tr_units)   # unit mask → die mask
        vl_mask  = np.isin(uid_train_die, vl_units)

        model = fit_zit(make_zit(spec, zeta_star),
                        X_train[tr_mask], y_train_die[tr_mask], uid_train_die[tr_mask], is_bag)
        oof_die[vl_mask] = model.predict(X_train[vl_mask])   # (1-π)μ, 음수 clip은 predict 내부
        val_die  += model.predict(X_val)  / N_FOLDS
        test_die += model.predict(X_test) / N_FOLDS
    assert not np.isnan(oof_die).any(), f'{label}: OOF die 미커버 — fold 누락'
    print(f'  [refit] {time.time()-t1:.0f}s')

    zit_cache[label] = {
        'zeta_star': zeta_star,
        'oof':  unit_agg(uid_train_die, oof_die,  agg),
        'val':  unit_agg(uid_val_die,   val_die,  agg),
        'test': unit_agg(uid_test_die,  test_die, agg),
    }

print(f'\n[ZIT 4종 완료] {time.time()-t_all:.0f}s')


===== zit_pearson (bag=False, agg=mean, phi=pearson) =====
  [ζ*] 1.60 (profile, train loglik 최대) | 987s
  [refit] 615s

===== zit_eql (bag=False, agg=mean, phi=eql) =====
  [ζ*] 1.10 (profile, train loglik 최대) | 942s
  [refit] 627s

===== bagzit_pearson (bag=True, agg=sum, phi=pearson) =====
  [ζ*] 1.60 (zit_pearson ζ* 차용)
  [refit] 615s

===== bagzit_eql (bag=True, agg=sum, phi=eql) =====
  [ζ*] 1.10 (zit_eql ζ* 차용)
  [refit] 634s

[ZIT 4종 완료] 4421s


## 4c. ZIT 백엔드 믹스 12종 — CatBoost(π/μ) × LGBM(φ), default

기존 ZIT 4 base(zit_pearson/zit_eql/bagzit_pearson/bagzit_eql)에 **π·μ 백엔드를 섞은 3 combo**를 곱해 12종 추가 (φ는 모두 LGBM 고정 — CatBoost 네이티브 gamma 부재 회피). ZIT 총 16종.

- **3 combo**: `pi_cb`(π=CatBoost·μ=LGBM) / `mu_cb`(π=LGBM·μ=CatBoost) / `pi_mu_cb`(둘 다 CatBoost)
- **CatBoost 기본값**: μ=`Tweedie:variance_power=ζ*` + sample_weight, π=`CrossEntropy`(soft-label). 나머지 HP는 CatBoost 라이브러리 기본값(iterations 1000 등) — "각 모델 자기 기본값" 철학 그대로
- **ζ\* 차용**: §4b baseline의 ζ*(pearson 1.80 / eql 1.10) 재사용 — profile 생략(ζ는 backend 거의 무관, 비용 8× 절감)
- **구현**: `zit.py` 무수정. 노트북 로컬 `_MixedZITMixin` + concrete 4종이 `_m_step`(π·μ 분기)·`predict_components`(catboost π는 predict_proba)만 오버라이드, `_initialize`/fit/φ는 각 부모 상속(=baseline과 동일)
- **체크포인트+resume**: combo별 결과를 `mixed_ckpt/<label>.pkl`에 저장, 재실행 시 완료분 skip (Colab 세션 끊겨도 이어서)
- **⚠ 본 실행(아래 ~5-8h) 전에 smoke-test 셀 먼저 통과 확인** (CatBoost CrossEntropy soft-label·predict_proba 동작)

In [7]:
# (4c-def) ZIT 백엔드 믹스 정의 — π/μ만 lgbm↔catboost 분기, φ는 항상 LGBM(부모 _fit_lgb_phi)
import lightgbm as lgb
from catboost import CatBoostRegressor, CatBoostClassifier
from modules.zit import _tweedie_unit_deviance        # EQL φ 타깃 계산용


class _MixedZITMixin:
    """π/μ 백엔드를 섞는 믹스인 (φ는 항상 LGBM 고정 → CatBoost gamma 부재 회피).

    추가 인자: pi_backend/mu_backend ('lgbm'|'catboost'), phi_method ('pearson'|'eql').
    _initialize/fit/predict_unit/score_loglik 등은 각 concrete 클래스의 부모(ZIT/EQL/Bag…)에서 상속.
    오버라이드는 _m_step(π·μ 백엔드 분기, φ 타깃은 phi_method로)과 predict_components(catboost π는 predict_proba)뿐.
    """

    def __init__(self, *, pi_backend='lgbm', mu_backend='lgbm', phi_method='pearson', **kwargs):
        super().__init__(**kwargs)            # 부모 ZITboostRegressor.__init__ 으로 나머지 HP 전달
        self.pi_backend = pi_backend
        self.mu_backend = mu_backend
        self.phi_method = phi_method

    def _cat_common(self):
        # CatBoost '기본값' — 재현/스레드/로그억제만 (iterations·depth 등은 라이브러리 기본값)
        tc = self.n_jobs if (self.n_jobs and self.n_jobs > 0) else -1
        return dict(random_seed=self.random_state, thread_count=tc,
                    verbose=False, allow_writing_files=False)

    def _fit_pi(self, X, posterior):
        if self.pi_backend == 'catboost':
            m = CatBoostClassifier(loss_function='CrossEntropy', **self._cat_common())
            m.fit(X, posterior)                          # soft label(0~1) 직접 학습
            pi_pred = m.predict_proba(X)[:, 1]
        else:
            m = lgb.LGBMRegressor(**self._pi_params())   # 부모의 cross_entropy 파라미터
            m.fit(X, posterior)
            pi_pred = m.predict(X)
        return m, np.clip(pi_pred, 1e-8, 1 - 1e-8)

    def _fit_mu(self, X, y, mu_weight):
        if self.mu_backend == 'catboost':
            m = CatBoostRegressor(loss_function=f'Tweedie:variance_power={self.zeta}',
                                  **self._cat_common())
            m.fit(X, y, sample_weight=mu_weight)
        else:
            m = lgb.LGBMRegressor(**self._mu_params())   # 부모의 tweedie 파라미터
            m.fit(X, y, sample_weight=mu_weight)
        return m, np.maximum(m.predict(X), 1e-10)

    def _m_step(self, X, y, posterior):
        w_tw = 1.0 - posterior
        # 1) π — soft label Π 회귀
        m_pi, pi_pred = self._fit_pi(X, posterior)
        # 2) μ — Tweedie, weight=(1-Π)/φ̂ (직전 φ)
        phi_for_weight = np.maximum(self._phi_current, 1e-10)
        m_mu, mu_pred = self._fit_mu(X, y, w_tw / phi_for_weight)
        # 3) φ — 항상 LGBM, 타깃만 phi_method로 분기 (부모 _m_step과 동일 수식)
        if self.phi_method == 'eql':
            phi_target = _tweedie_unit_deviance(y, mu_pred, self.zeta)
        else:
            resid_sq = np.square(y - mu_pred)
            phi_target = np.clip(resid_sq / np.maximum(np.power(mu_pred, self.zeta), 1e-10), 1e-8, 1e6)
        m_phi = self._fit_lgb_phi(X, phi_target, w_tw)   # 퇴화 split 가드 포함(부모)
        phi_pred = np.clip(m_phi.predict(X), 1e-8, 1e6)
        return m_pi, m_mu, m_phi, pi_pred, mu_pred, phi_pred

    def predict_components(self, X):
        if not hasattr(self, 'fitted_'):
            raise ValueError('Model not fitted. Call fit() first.')
        X = np.asarray(X, dtype=np.float64)
        if self.pi_backend == 'catboost':
            pi = self.lgb_pi_.predict_proba(X)[:, 1]     # CatBoostClassifier → 확률
        else:
            pi = self.lgb_pi_.predict(X)                 # LGBM cross_entropy = 확률
        pi = np.clip(pi, 1e-8, 1 - 1e-8)
        mu = np.maximum(self.lgb_mu_.predict(X), 1e-10)
        phi = np.clip(self.lgb_phi_.predict(X), 1e-8, 1e6)
        return pi, mu, phi


# concrete 4종 — _initialize는 각 부모 그대로(=baseline과 일치), _m_step/predict_components는 믹스인이 오버라이드
class ZITPearsonMixed(_MixedZITMixin, ZITboostRegressor):       pass   # non-bag, pearson init
class ZITEqlMixed(_MixedZITMixin, ZITboostEQLRegressor):        pass   # non-bag, EQL init
class BagZITPearsonMixed(_MixedZITMixin, BagZITboostRegressor): pass   # bag, pearson init
class BagZITEqlMixed(_MixedZITMixin, BagZITEQLRegressor):       pass   # bag, pearson init(=baseline)


def make_zit_mixed(cls, zeta, phi_method, pi_backend, mu_backend):
    # baseline make_zit 와 동일 인자 + 백엔드/φ방식. LGBM 컴포넌트는 LGBM_DEFAULTS(라이브러리 기본값)
    return cls(
        zeta=zeta, n_em_iters=N_EM_ITERS,
        pi_backend=pi_backend, mu_backend=mu_backend, phi_method=phi_method,
        random_state=SEED, n_jobs=N_JOBS, verbose=-1, device='cpu', **LGBM_DEFAULTS,
    )


# 4 base × 3 backend combo = 12종 (φ는 모두 LGBM)
_MIXED_BASE = [
    dict(label='zit_pearson',    cls=ZITPearsonMixed,    bag=False, agg='mean', phi='pearson'),
    dict(label='zit_eql',        cls=ZITEqlMixed,        bag=False, agg='mean', phi='eql'),
    dict(label='bagzit_pearson', cls=BagZITPearsonMixed, bag=True,  agg='sum',  phi='pearson'),
    dict(label='bagzit_eql',     cls=BagZITEqlMixed,     bag=True,  agg='sum',  phi='eql'),
]
_BACKEND_COMBOS = [
    dict(tag='pi_cb',    pi_backend='catboost', mu_backend='lgbm'),      # π=CatBoost, μ=LGBM
    dict(tag='mu_cb',    pi_backend='lgbm',     mu_backend='catboost'),  # π=LGBM,     μ=CatBoost
    dict(tag='pi_mu_cb', pi_backend='catboost', mu_backend='catboost'),  # 둘 다 CatBoost
]
MIXED_SPECS = [
    dict(label=f"{b['label']}__{c['tag']}", cls=b['cls'], bag=b['bag'], agg=b['agg'], phi=b['phi'],
         pi_backend=c['pi_backend'], mu_backend=c['mu_backend'])
    for b in _MIXED_BASE for c in _BACKEND_COMBOS
]
print(f'MIXED_SPECS: {len(MIXED_SPECS)}종 →', [s['label'] for s in MIXED_SPECS])

MIXED_SPECS: 12종 → ['zit_pearson__pi_cb', 'zit_pearson__mu_cb', 'zit_pearson__pi_mu_cb', 'zit_eql__pi_cb', 'zit_eql__mu_cb', 'zit_eql__pi_mu_cb', 'bagzit_pearson__pi_cb', 'bagzit_pearson__mu_cb', 'bagzit_pearson__pi_mu_cb', 'bagzit_eql__pi_cb', 'bagzit_eql__mu_cb', 'bagzit_eql__pi_mu_cb']


In [8]:
# (4c-smoke) CatBoost π/μ 컴포넌트가 EM 안에서 정상 동작하는지 소량(3k행)·2 iter로 검증
#   ⚠ 본 실행(다음 셀, ~5-8h) 전에 이 셀이 통과해야 함 — CatBoost CrossEntropy soft-label·predict_proba 확인
_n = 3000
_Xs, _ys, _us = X_train[:_n], y_train_die[:_n], uid_train_die[:_n]
_cases = [
    ('zit_pearson__pi_mu_cb', ZITPearsonMixed, False, 'pearson', 'catboost', 'catboost'),
    ('bagzit_eql__pi_cb',     BagZITEqlMixed,  True,  'eql',     'catboost', 'lgbm'),
    ('zit_eql__mu_cb',        ZITEqlMixed,     False, 'eql',     'lgbm',     'catboost'),
]
for _tag, _cls, _bag, _phi, _pib, _mub in _cases:
    _m = make_zit_mixed(_cls, 1.5, _phi, _pib, _mub)
    _m.n_em_iters = 2                            # smoke: 2 iter만
    if _bag:
        _m.fit(_Xs, _ys, unit_id=_us)
    else:
        _m.fit(_Xs, _ys)
    _p = _m.predict(_Xs)
    _pi, _mu, _phi_ = _m.predict_components(_Xs)
    assert _p.shape == (_n,) and np.all(_p >= 0),  f'{_tag}: pred 형태/음수 이상'
    assert np.all((_pi > 0) & (_pi < 1)),          f'{_tag}: π 범위 이상'
    assert np.all(_mu > 0) and np.all(_phi_ > 0),  f'{_tag}: μ/φ 비양수'
    print(f'[smoke OK] {_tag}: pred[{_p.min():.4g}, {_p.max():.4g}] · π_mean={_pi.mean():.3f}')
print('\nsmoke test 통과 — 다음 셀(본 실행) 진행 가능')

[smoke OK] zit_pearson__pi_mu_cb: pred[5.81e-05, 0.01379] · π_mean=0.826
[smoke OK] bagzit_eql__pi_cb: pred[4.577e-05, 0.018] · π_mean=0.823
[smoke OK] zit_eql__mu_cb: pred[1.186e-05, 0.01488] · π_mean=0.826

smoke test 통과 — 다음 셀(본 실행) 진행 가능


In [9]:
# (4c) ZIT 믹스 12종 — ζ* 차용(profile 생략) + combo별 체크포인트/resume + 5-fold refit
import pickle

# ζ* 차용: §4b(c10) all-LGBM baseline에서 결정된 ζ*(pearson/eql) 재사용 (셀 단독 재실행 대비 폴백 포함)
try:
    ZETA_STAR = dict(zeta_by_phi)
except NameError:
    ZETA_STAR = {'pearson': 1.80, 'eql': 1.10}
print('[ζ* 차용]', {k: round(v, 2) for k, v in ZETA_STAR.items()})

CKPT_DIR = os.path.join(OUT_DIR, 'mixed_ckpt')
os.makedirs(CKPT_DIR, exist_ok=True)

t_all = time.time()
for spec in MIXED_SPECS:
    label, is_bag, agg, phi = spec['label'], spec['bag'], spec['agg'], spec['phi']
    pib, mub = spec['pi_backend'], spec['mu_backend']
    cp = os.path.join(CKPT_DIR, f'{label}.pkl')

    if os.path.exists(cp):                       # resume: 완료된 combo는 로드 후 skip
        with open(cp, 'rb') as f:
            zit_cache[label] = pickle.load(f)
        zs = zit_cache[label]['zeta_star']
        print(f'[skip] {label} (ckpt 로드, ζ*={zs:.2f})')
        continue

    zeta_star = ZETA_STAR[phi]
    print(f'\n===== {label} (bag={is_bag}, phi={phi}, pi={pib}, mu={mub}, ζ*={zeta_star:.2f}) =====')

    oof_die  = np.full(len(X_train), np.nan)
    val_die  = np.zeros(len(X_val))
    test_die = np.zeros(len(X_test))
    t1 = time.time()
    for tr_uidx, vl_uidx in FOLDS:
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask  = np.isin(uid_train_die, tr_units)   # unit mask → die mask
        vl_mask  = np.isin(uid_train_die, vl_units)

        model = fit_zit(
            make_zit_mixed(spec['cls'], zeta_star, phi, pib, mub),
            X_train[tr_mask], y_train_die[tr_mask], uid_train_die[tr_mask], is_bag,
        )
        oof_die[vl_mask] = model.predict(X_train[vl_mask])   # (1-π)μ die-level, 음수 clip 내부
        val_die  += model.predict(X_val)  / N_FOLDS
        test_die += model.predict(X_test) / N_FOLDS
    assert not np.isnan(oof_die).any(), f'{label}: OOF die 미커버 — fold 누락'

    rec = {
        'zeta_star': zeta_star,
        'oof':  unit_agg(uid_train_die, oof_die,  agg),   # 일반 ZIT=mean / BagZIT=sum
        'val':  unit_agg(uid_val_die,   val_die,  agg),
        'test': unit_agg(uid_test_die,  test_die, agg),
    }
    zit_cache[label] = rec
    with open(cp, 'wb') as f:                    # 체크포인트 저장 (다음 실행 시 skip)
        pickle.dump(rec, f)
    print(f'  [refit] {time.time()-t1:.0f}s · ckpt 저장: {os.path.basename(cp)}')

ZIT_ALL_SPECS = ZIT_SPECS + MIXED_SPECS          # §5(c14)가 16종 전체를 집계
print(f'\n[ZIT 믹스 12종 완료] {time.time()-t_all:.0f}s · ZIT 총 {len(ZIT_ALL_SPECS)}종')

[ζ* 차용] {'pearson': np.float64(1.6), 'eql': np.float64(1.1)}

===== zit_pearson__pi_cb (bag=False, phi=pearson, pi=catboost, mu=lgbm, ζ*=1.60) =====
  [refit] 4402s · ckpt 저장: zit_pearson__pi_cb.pkl

===== zit_pearson__mu_cb (bag=False, phi=pearson, pi=lgbm, mu=catboost, ζ*=1.60) =====
  [refit] 2668s · ckpt 저장: zit_pearson__mu_cb.pkl

===== zit_pearson__pi_mu_cb (bag=False, phi=pearson, pi=catboost, mu=catboost, ζ*=1.60) =====
  [refit] 6133s · ckpt 저장: zit_pearson__pi_mu_cb.pkl

===== zit_eql__pi_cb (bag=False, phi=eql, pi=catboost, mu=lgbm, ζ*=1.10) =====
  [refit] 4128s · ckpt 저장: zit_eql__pi_cb.pkl

===== zit_eql__mu_cb (bag=False, phi=eql, pi=lgbm, mu=catboost, ζ*=1.10) =====
  [refit] 2535s · ckpt 저장: zit_eql__mu_cb.pkl

===== zit_eql__pi_mu_cb (bag=False, phi=eql, pi=catboost, mu=catboost, ζ*=1.10) =====
  [refit] 6009s · ckpt 저장: zit_eql__pi_mu_cb.pkl

===== bagzit_pearson__pi_cb (bag=True, phi=pearson, pi=catboost, mu=lgbm, ζ*=1.60) =====
  [refit] 4092s · ckpt 저장: bagzit_pea

## 4d. zit_gu — Gu 2024 Algorithm 1 충실판 (인라인, 모듈 무수정)

노트북이 쓰는 `zit_eql`(=`ZITboostEQLRegressor`)은 논문 방향이지만 **Algorithm 1 Step1의 초기화 GBT를 생략**한 근사판이다. 여기서는 그 단계를 채운 **Gu 충실판**을 `ZITboostEQLRegressor` 상속으로 인라인 정의한다 (`zit.py`/`zit_Gu.py` **무수정**).

- **Gu 교정 2점만 오버라이드** (나머지 `__init__`/μ/π 파라미터/fit/predict/score_loglik은 부모 상속):
  - ① `_initialize`: 스칼라 초기값 위에 초기 GBT F̂⁰_π/μ/φ 를 적합(Alg.1 Step1) → 첫 E-step posterior가 상수가 아닌 GBT 기반
  - ② φ M-step: 논문 eq.(8) shape=½ gamma(log-link)를 custom objective로 직접. **LightGBM 4.x는 `lgb.train(fobj=)`가 제거 → `params={'objective': callable}`로 전달** (모듈 `zit_Gu.py`의 LightGBM 3.x `fobj` API가 4.x에서 깨지는 문제 회피)
- **적용 범위**: non-bag·base 단독. bag/§4c 백엔드 믹스에는 미적용 (φ가 log-link Booster라 믹스의 `_fit_lgb_phi` 가정과 불일치)
- **비교 가치**: 같은 fold·전처리에서 `zit_eql` vs `zit_gu` → "초기화 GBT가 RMSE를 바꾸는가"를 직접 확인. φ는 `reg_lambda=0`이라 eq.(8) custom-obj와 stock-gamma가 트리상 동일 → 실측 차이는 사실상 ①(초기화 GBT)에서 나옴
- **런타임 검증**: 합성 zero-inflated 데이터로 fit/predict/score_loglik + `n_jobs∈{10,-1}` PASS, `zit_eql`과 예측 상이(=초기화 GBT 효과) 확인 완료

In [10]:
# 4d. zit_gu 클래스 정의 — ZITboostEQLRegressor 상속, Gu 교정 2점만 오버라이드 (zit.py/zit_Gu.py 무수정)
import lightgbm as lgb
from modules.zit import _tweedie_unit_deviance   # 부모와 동일 deviance(대수적으로 zit_Gu의 식과 동일)


class ZITGuInline(ZITboostEQLRegressor):
    """Gu 2024(arXiv:2405.14990) Algorithm 1 충실판 — 노트북 인라인(모듈 무수정).

    부모 ZITboostEQLRegressor(논문방향 EQL) 대비 Gu가 추가로 교정하는 2점만 오버라이드:
      ① _initialize : 스칼라 초기값 위에 초기 GBT F̂^(0)_π/μ/φ 를 적합(Alg.1 Step1).
                      (부모 EQL은 이 단계를 생략 → 첫 E-step posterior가 상수)
      ② φ M-step    : 논문 eq.(8) shape=½ gamma(log-link)를 custom objective로 직접 구현.
                      LightGBM 4.x에서 lgb.train(fobj=)가 제거돼 params={'objective': callable}로 전달.
    __init__/μ/π 파라미터/fit/predict/score_loglik 은 부모 그대로 상속.
    """

    @staticmethod
    def _phi_custom_obj(d_w, w_arr):
        # eq.(8) shape=1/2 gamma, log-link φ=exp(F). grad/hess (LightGBM은 loss 최소화 → 부호 반영).
        def _obj(preds, dataset):
            phi = np.maximum(np.exp(preds), 1e-10)
            grad = w_arr * 0.5 - d_w / (2.0 * phi)        # (1-Π)·½ − (1-Π)·D/(2φ)
            hess = np.maximum(d_w / (2.0 * phi), 1e-8)    # 양수 클램프
            return grad, hess
        return _obj

    def _fit_phi_custom(self, X, d_w, w_arr):
        # d_w=(1-Π)·D_ζ, w_arr=(1-Π). init_score=log(mean(D))로 log-link 공간에서 시작.
        d_mean = float(np.mean(d_w / np.maximum(w_arr, 1e-10)))
        init_score = np.full(len(X), np.log(max(d_mean, 1e-10)))
        md = self.phi_max_depth if (self.phi_max_depth and self.phi_max_depth > 0) else -1

        def _train(min_child, max_depth):
            ds = lgb.Dataset(X, init_score=init_score, free_raw_data=False)
            return lgb.train(
                params={"objective": self._phi_custom_obj(d_w, w_arr),   # 4.x: params에 callable
                        "num_leaves": self.phi_num_leaves, "max_depth": max_depth,
                        "min_child_samples": min_child, "learning_rate": self.phi_learning_rate,
                        "num_iterations": self.phi_n_estimators, "seed": self.random_state,
                        "num_threads": self.n_jobs, "device": self.device, "verbosity": -1},
                train_set=ds)

        try:
            return _train(self.phi_min_child_samples, md)
        except lgb.basic.LightGBMError:                  # 퇴화 split 가드 (부모 _fit_lgb_phi와 동일 정책)
            mc = max(int(self.phi_min_child_samples or 20), 100)
            return _train(mc, 8 if md <= 0 else min(md, 8))

    def _initialize(self, X, y):
        # 스칼라 초기값(부모 EQL과 동일) + Alg.1 Step1 초기 GBT 3개
        n = len(y); is_zero = (y == 0); is_pos = ~is_zero; n_pos = int(is_pos.sum())
        mu0 = max(float(y[is_pos].mean()) if n_pos > 0 else 1e-4, 1e-10)
        if n_pos > 0:
            dev = _tweedie_unit_deviance(y[is_pos], np.full(n_pos, mu0), self.zeta)
            phi0 = float(np.clip(dev.sum() / n_pos, 1e-6, 1e6))      # zero-truncated deviance moment
        else:
            phi0 = 1.0
        pi0 = float(np.clip(is_zero.mean(), 0.01, 0.99))

        lp = lgb.LGBMRegressor(**self._pi_params()); lp.fit(X, np.full(n, pi0))            # F̂^(0)_π
        pi_arr = np.clip(lp.predict(X), 1e-8, 1 - 1e-8)
        lm = lgb.LGBMRegressor(**self._mu_params())                                        # F̂^(0)_μ (eq.7 구조)
        lm.fit(X, y, sample_weight=np.full(n, (1 - pi0) / phi0))
        mu_arr = np.maximum(lm.predict(X), 1e-10)
        dev0 = _tweedie_unit_deviance(y, np.full(n, mu0), self.zeta)
        bphi0 = self._fit_phi_custom(X, (1 - pi0) * dev0, np.full(n, 1 - pi0))             # F̂^(0)_φ (eq.8 구조)
        phi_arr = np.clip(np.exp(bphi0.predict(X)), 1e-8, 1e6)
        return pi_arr, mu_arr, phi_arr

    def _m_step(self, X, y, posterior):
        # 부모 fit이 self._phi_current를 세팅하고 _m_step(X,y,posterior)로 호출 → 부모와 동일 시그니처
        one_minus = 1.0 - posterior
        lgb_pi = lgb.LGBMRegressor(**self._pi_params()); lgb_pi.fit(X, posterior)          # eq.6
        pi_pred = np.clip(lgb_pi.predict(X), 1e-8, 1 - 1e-8)
        mu_w = one_minus / np.maximum(self._phi_current, 1e-10)                            # eq.7 weight (1-Π)/φ̂
        lgb_mu = lgb.LGBMRegressor(**self._mu_params()); lgb_mu.fit(X, y, sample_weight=mu_w)
        mu_pred = np.maximum(lgb_mu.predict(X), 1e-10)
        dz = _tweedie_unit_deviance(y, mu_pred, self.zeta)                                 # eq.8 custom-obj
        bphi = self._fit_phi_custom(X, one_minus * dz, one_minus)
        phi_pred = np.clip(np.exp(bphi.predict(X)), 1e-8, 1e6)
        return lgb_pi, lgb_mu, bphi, pi_pred, mu_pred, phi_pred

    def predict_components(self, X):
        if not hasattr(self, "fitted_"):
            raise ValueError("Model not fitted. Call fit() first.")
        X = np.asarray(X, dtype=np.float64)
        pi = np.clip(self.lgb_pi_.predict(X), 1e-8, 1 - 1e-8)
        mu = np.maximum(self.lgb_mu_.predict(X), 1e-10)
        phi = np.clip(np.exp(self.lgb_phi_.predict(X)), 1e-8, 1e6)   # custom-obj는 log(φ) 예측 → exp 역변환
        return pi, mu, phi


print("ZITGuInline 정의 완료 (ZITboostEQLRegressor 상속, _initialize/_m_step/predict_components/φ-obj 오버라이드)")

ZITGuInline 정의 완료 (ZITboostEQLRegressor 상속, _initialize/_m_step/predict_components/φ-obj 오버라이드)


In [11]:
# 4d. zit_gu fit — ζ profile(8) → 5-fold refit (c10 non-bag 경로와 동일, spec 1개만). 기존 zit_cache 보존.
import time as _time
GU_SPEC = {'label': 'zit_gu', 'cls': ZITGuInline, 'bag': False, 'agg': 'mean', 'phi': 'eql_gu'}

# (1) ζ profile likelihood — non-bag이므로 c10과 동일하게 train loglik 최대 ζ* (zeta_by_phi는 건드리지 않음)
_t0 = _time.time()
_ll = {}
for _z in ZETA_GRID:
    _m = fit_zit(make_zit(GU_SPEC, _z), X_train, y_train_die, uid_train_die, False)
    _ll[_z] = _m.score_loglik(X_train, y_train_die)
gu_zeta_star = max(_ll, key=_ll.get)
print(f'[zit_gu ζ*] {gu_zeta_star:.2f} (profile, train loglik 최대) | {_time.time()-_t0:.0f}s')

# (2) ζ* 5-fold refit — die (1-π)μ → oof/val/test (val/test는 fold 평균), 일반 ZIT와 동일 unit mean 집계
_oof = np.full(len(X_train), np.nan); _val = np.zeros(len(X_val)); _test = np.zeros(len(X_test))
_t1 = _time.time()
for _tr, _vl in FOLDS:
    _tru, _vlu = unique_units[_tr], unique_units[_vl]
    _trm = np.isin(uid_train_die, _tru); _vlm = np.isin(uid_train_die, _vlu)
    _mdl = fit_zit(make_zit(GU_SPEC, gu_zeta_star),
                   X_train[_trm], y_train_die[_trm], uid_train_die[_trm], False)
    _oof[_vlm] = _mdl.predict(X_train[_vlm])      # (1-π)μ, 음수 clip은 predict 내부
    _val  += _mdl.predict(X_val)  / N_FOLDS
    _test += _mdl.predict(X_test) / N_FOLDS
assert not np.isnan(_oof).any(), 'zit_gu: OOF die 미커버 — fold 누락'
zit_cache['zit_gu'] = {
    'zeta_star': gu_zeta_star,
    'oof':  unit_agg(uid_train_die, _oof,  'mean'),
    'val':  unit_agg(uid_val_die,   _val,  'mean'),
    'test': unit_agg(uid_test_die,  _test, 'mean'),
}
print(f'[zit_gu refit] {_time.time()-_t1:.0f}s')

# (3) §5(c14)가 읽는 ZIT_ALL_SPECS에 등록 (중복 방지). §4c 미실행 시 ZIT_SPECS 기반으로 새로 생성.
if 'ZIT_ALL_SPECS' not in globals():
    ZIT_ALL_SPECS = list(ZIT_SPECS)
if not any(s['label'] == 'zit_gu' for s in ZIT_ALL_SPECS):
    ZIT_ALL_SPECS.append(GU_SPEC)
print('ZIT_ALL_SPECS:', [s['label'] for s in ZIT_ALL_SPECS])

[zit_gu ζ*] 1.50 (profile, train loglik 최대) | 1599s
[zit_gu refit] 1054s
ZIT_ALL_SPECS: ['zit_pearson', 'zit_eql', 'bagzit_pearson', 'bagzit_eql', 'zit_pearson__pi_cb', 'zit_pearson__mu_cb', 'zit_pearson__pi_mu_cb', 'zit_eql__pi_cb', 'zit_eql__mu_cb', 'zit_eql__pi_mu_cb', 'bagzit_pearson__pi_cb', 'bagzit_pearson__mu_cb', 'bagzit_pearson__pi_mu_cb', 'bagzit_eql__pi_cb', 'bagzit_eql__mu_cb', 'bagzit_eql__pi_mu_cb', 'zit_gu']


## 5. 결과 비교 + 저장 — results.csv (46행)

세 모드의 die/unit 예측(`cache`·`zit_cache`)을 unit RMSE로 집계해 한 표로 합치고 **한 번만 저장**한다.
- 기본 회귀 5건 · 투스테이지 25건(clf 5 × reg 5, die 곱 → unit mean) · ZIT 16건(4 base + 백엔드 믹스 12) = **46행**
- 후처리 없음 (τ/position/zero_clip/집계선택 미적용)
- §4c 미실행 시 ZIT 4건만 집계(34행)로 자동 폴백

In [12]:
# 기본 회귀 RMSE (5건) — die 예측 → unit mean → RMSE
rows = []
for name in MODELS:
    die = cache[name]['basic']
    r = {sp: rmse_unit(unit_agg(UID[sp], die[sp], 'mean'), YK[sp]) for sp in ('oof', 'val', 'test')}
    rows.append({'mode': 'reg', 'clf': '-', 'reg': name,
                 'oof_rmse': r['oof'], 'val_rmse': r['val'], 'test_rmse': r['test']})

reg_df = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
print('=== 기본 회귀 (val 오름차순) ===')
print(reg_df.to_string(index=False, float_format='%.6f'))

=== 기본 회귀 (val 오름차순) ===
mode clf      reg  oof_rmse  val_rmse  test_rmse
 reg   - catboost  0.005540  0.005733   0.008432
 reg   -     lgbm  0.005540  0.005735   0.008434
 reg   -      xgb  0.005576  0.005741   0.008434
 reg   -       et  0.005577  0.005762   0.008446
 reg   -       rf  0.005589  0.005770   0.008452


In [13]:
# 투스테이지 그리드 RMSE (clf 5 × reg 5 = 25건) — die-level P(Y>0) × E[Y|Y>0] 곱 → unit mean → RMSE (후처리 없음)
grid_rows = []
for c in MODELS:                 # 분류기
    proba = cache[c]['clf']
    for r in MODELS:             # 회귀기
        regd = cache[r]['tsreg']
        res = {}
        for sp in ('oof', 'val', 'test'):
            final_die = np.asarray(proba[sp]) * np.asarray(regd[sp])   # die 단위 곱
            res[sp] = rmse_unit(unit_agg(UID[sp], final_die, 'mean'), YK[sp])
        grid_rows.append({'mode': 'two_stage', 'clf': c, 'reg': r,
                          'oof_rmse': res['oof'], 'val_rmse': res['val'], 'test_rmse': res['test']})

ts_df = pd.DataFrame(grid_rows).sort_values('val_rmse').reset_index(drop=True)
print('=== 투스테이지 그리드 (val 오름차순) ===')
print(ts_df.to_string(index=False, float_format='%.6f'))

=== 투스테이지 그리드 (val 오름차순) ===
     mode      clf      reg  oof_rmse  val_rmse  test_rmse
two_stage     lgbm     lgbm  0.005523  0.005718   0.008418
two_stage      xgb     lgbm  0.005539  0.005719   0.008413
two_stage     lgbm catboost  0.005526  0.005721   0.008419
two_stage catboost     lgbm  0.005525  0.005722   0.008418
two_stage     lgbm      xgb  0.005543  0.005722   0.008421
two_stage      xgb catboost  0.005541  0.005722   0.008413
two_stage      xgb      xgb  0.005559  0.005722   0.008416
two_stage catboost catboost  0.005528  0.005724   0.008418
two_stage     lgbm       rf  0.005526  0.005724   0.008417
two_stage catboost      xgb  0.005545  0.005724   0.008421
two_stage     lgbm       et  0.005525  0.005724   0.008420
two_stage      xgb       rf  0.005542  0.005725   0.008411
two_stage      xgb       et  0.005542  0.005726   0.008415
two_stage catboost       rf  0.005528  0.005727   0.008416
two_stage catboost       et  0.005527  0.005728   0.008420
two_stage       rf     lgbm

In [14]:
# ZIT 전체 RMSE — die 예측은 §4b/§4c에서 이미 unit 집계됨(일반=mean/Bag=sum)
#   §4c(믹스 12종) 실행 시 ZIT_ALL_SPECS(16종), 미실행 시 ZIT_SPECS(4종)로 폴백
_zit_specs = ZIT_ALL_SPECS if 'ZIT_ALL_SPECS' in globals() else ZIT_SPECS
zit_rows = []
for spec in _zit_specs:
    label = spec['label']
    c = zit_cache[label]
    zit_rows.append({
        'mode': 'zit', 'clf': '-',
        'reg': f"{label}_z{c['zeta_star']:.2f}",   # ζ*는 reg 라벨에 기록
        'oof_rmse':  rmse_unit(c['oof'],  y_train_unit_s),
        'val_rmse':  rmse_unit(c['val'],  y_val_unit_s),
        'test_rmse': rmse_unit(c['test'], y_test_unit_s),
    })

zit_df = pd.DataFrame(zit_rows).sort_values('val_rmse').reset_index(drop=True)
print(f'=== ZIT {len(zit_df)}종 (4 base + backend-mix, val 오름차순) ===')
print(zit_df.to_string(index=False, float_format='%.6f'))

# reg(5) + two_stage(25) + zit(4 or 16) 통합 저장 (단일 저장)
results = pd.concat([reg_df, ts_df, zit_df], ignore_index=True)
out_path = os.path.join(OUT_DIR, 'results.csv')
results.to_csv(out_path, index=False)
print(f'\n저장: {out_path}  ({len(results)}행)')

print('\n=== 전체 통합 (val 오름차순, head 15) ===')
print(results.sort_values('val_rmse').head(15).to_string(index=False, float_format='%.6f'))

=== ZIT 17종 (4 base + backend-mix, val 오름차순) ===
mode clf                            reg  oof_rmse  val_rmse  test_rmse
 zit   -              zit_pearson_z1.60  0.005548  0.005715   0.008415
 zit   -           bagzit_pearson_z1.60  0.005562  0.005718   0.008414
 zit   -       zit_pearson__pi_cb_z1.60  0.005537  0.005718   0.008416
 zit   -           zit_eql__mu_cb_z1.10  0.005508  0.005720   0.008422
 zit   -                  zit_eql_z1.10  0.005515  0.005722   0.008424
 zit   -        zit_eql__pi_mu_cb_z1.10  0.005511  0.005724   0.008426
 zit   -    zit_pearson__pi_mu_cb_z1.60  0.005521  0.005725   0.008425
 zit   -           zit_eql__pi_cb_z1.10  0.005515  0.005725   0.008425
 zit   -    bagzit_pearson__pi_cb_z1.60  0.005522  0.005728   0.008429
 zit   -               bagzit_eql_z1.10  0.005531  0.005730   0.008430
 zit   -        bagzit_eql__mu_cb_z1.10  0.005517  0.005730   0.008432
 zit   -     bagzit_eql__pi_mu_cb_z1.10  0.005521  0.005731   0.008433
 zit   -                   z